# 対象の動詞は上下の移動を表すか

上から順にセルを実行し、規則を変え、分類器を学習させて、不一致を一つ読みます。暫定ラベルはQwen3.5-9Bによるものです。上下の移動が起きる・起きた場合はCT+、否定される場合はCT-、起きるか示されていない場合はUu、上下の移動を表さない場合はNMです。 自分の予想と本文の読みは、結果を見る前にメモしてください。PythonのエラーはAIに質問してかまいません。

### 教材データを読み込む
準備セルを一度実行してください。箇所のIDで本文・暫定ラベル・埋め込みを対応付けます。自分で変更するのは「編集」と書かれた二つのセルです。

In [ ]:
#@title 本文・ラベル・埋め込みを読み込む
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

url = "https://nlp.lang.osaka-u.ac.jp/dh/b/materials/movement-classification.zip"
archive, _ = urlretrieve(url)
with ZipFile(archive) as bundle:
    passages = pd.read_json(bundle.open("class7/passages.jsonl"), lines=True).set_index("id")
    labels = pd.read_json(bundle.open("class7/llm-labels.jsonl"), lines=True).set_index("id")
    with np.load(bundle.open("class7/embeddings.npz")) as saved:
        vectors = pd.DataFrame(saved["vectors"].astype("float32"), index=saved["ids"])

passages["reference"] = labels["label"]

train = passages[passages["split"] == "train"]
dev = passages[passages["split"] == "dev"].dropna(subset=["reference"])
test = passages[passages["split"] == "test"].dropna(subset=["reference"])
train_vectors = vectors.loc[train.index]
train_labels = train["reference"]

def show_passage(row):
    print(row["author"], row["title"], row["sentence"])
    print(row["previous"])
    print(row["left"] + "【" + row["target"] + "】" + row["right"])
    print(row["next"])

### 1. 予測の前に読む
【対象語】と前後の文を読んでください。何が落ちるのでしょうか。次に進む前にラベルを一つ選んでください。

In [ ]:
passage = dev.loc["dev-0054"]
show_passage(passage)

**編集1**　「落ちる」の直前が手掛かりの語句で終わればNM、それ以外ならCT+とする規則です。Pythonの `left.endswith(cue)` で、左文脈がその語句で終わるかを調べます。上の箇所でどう予測するか、メモしてから次の二つのセルを実行してください。次に「眠りに」を、本文で何が落ちるかを示す「が」で終わる語句に置き換え、同じ二つのセルを実行してください。

In [ ]:
cue = "眠りに"

In [ ]:
left = passage["left"]
if left.endswith(cue):
    prediction = "NM"
else:
    prediction = "CT+"
print('規則', prediction)
print('Qwen', passage["reference"])

同じ規則を「本国の土佐へ【落ち】て行った」に使うと、予測はどうなりますか。予測をメモしてから次のセルで前後の文を読み、自分のラベルを決めてください。上下の移動かどうかを判断する手掛かりはどの語句ですか。

In [ ]:
show_passage(dev.loc["dev-0046"])

Qwenの読みを開きます。自分の判断と違った場合は、どの語句から別の読みができるか説明してください。

In [ ]:
print("Qwen", dev.loc["dev-0046", "reference"])

### 2. 分類器を学習させる
ここからは、各箇所の埋め込みだけを入力に使います。学習用の420箇所には、それぞれ数値の列（埋め込み）と暫定ラベルがあります。`fit`でこの組み合わせを学習させます。次に、学習した分類器で別の作品のラベルを予測します。

In [ ]:
model = LogisticRegression(max_iter=2000)
model.fit(train_vectors, train_labels)
print(len(train_labels), '箇所で学習しました')

検証データで予測をQwenの読みと比べます。行はQwenのラベル、列は予測です。Uuの行を見ると、何件中何件を拾えましたか。CT-の行から、その再現率について何が言えるでしょうか。

In [ ]:
dev_predictions = model.predict(vectors.loc[dev.index])
matrix = pd.crosstab(dev["reference"], dev_predictions, colnames=["prediction"])
labels_in_order = ["CT+", "CT-", "Uu", "NM"]
matrix.reindex(index=labels_in_order, columns=labels_in_order, fill_value=0)

### 3. 評価データで不一致を読む
次のセルでQwenと一致した件数を数え、不一致の箇所を集めます。

In [ ]:
test_predictions = model.predict(vectors.loc[test.index])
results = test.assign(prediction=test_predictions)
agrees = results["prediction"] == results["reference"]
print(agrees.sum(), "/", len(results))
disagreements = results[~agrees]

**編集2**　まず0番の箇所を選びます。次の二つのセルを実行し、前後の文から対象語をどう読むか、自分のラベルと手掛かりになった語句を一つメモしてください。

In [ ]:
case_number = 0

In [ ]:
case = disagreements.iloc[case_number]
show_passage(case)

次のセルで二つのラベルを表示します。どの読みを本文から説明できますか。次に `case_number` を1に変え、番号のセルと本文のセルを再実行してください。別の箇所でも、ラベルを見る前に自分で判断してください。

In [ ]:
print("Qwen", case["reference"])
print('分類器', case["prediction"])